# Validation and Calibration on simulated data (following review)

In [ ]:
import jax
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
jax.config.update("jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir")

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from importlib import reload
import xarray as xr
import os, time, shutil
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "core").exists() and (REPO_ROOT.parent / "core").exists():
    os.chdir(REPO_ROOT.parent)
    REPO_ROOT = Path.cwd()
    
from core import data, config, pipelines, net, infer, train
reload(net); reload(train)
#from helpers import utils

if not os.path.exists('figs'):
    os.mkdir('figs')
if not os.path.exists('ckpts'):
    os.mkdir('ckpts')
    
config.Config.configure_AMIDE_clinical_whole_brain()    

In [ ]:
from core import train

output_dir = REPO_ROOT / "figs/simulative"
std_up_fact = 0.1
force_diag = False
num_test_voxels_x100 = 4

mt_sim_mode = 'isar2_c'
do_train = True
do_amide = False

# Adjust this value to increase/decrease noise in synthetic data
NOISE_LEVEL = list(np.arange(0.01, 0.04, 0.002))

do_test_bias = False
do_train_supervised = True

In [ ]:
print (f"std_up_fact: {std_up_fact}, force_diag: {force_diag}, mt_sim_mode: {mt_sim_mode}, do_train: {do_train}, do_amide: {do_amide}, output_dir: {output_dir}")

In [ ]:
from notebooks import nutils
reload(nutils)
nutils.config4sim()

train.train_config.std_up_fact = std_up_fact
infer.infer_config.force_diag = force_diag
pipelines.pipeline_config.mt_patience = 5000

In [ ]:
import datetime

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M") 

if False:  
    ## run with existing training results
    results_folder = Path('/home/alexf/phiVAE/figs/simulative/results_2026-06-01_14-17')
    do_train = False
else:  
    # normal path
    print(f"Timestamp for this run: {timestamp}")
    results_folder = Path(output_dir) / f"results_{timestamp}"
    os.makedirs(results_folder, exist_ok=True)

In [ ]:
ranges4synth = data.def_ranges4LHS
ranges4synth['T2a_ms'] = [50, 120] 
ranges4synth['B0_shift_ppm_map'] = [-0.3, 0.3] 
ranges4synth['B1_fix_factor_map'] = [0.9, 1.1] 
ranges4synth['fc_gt_T'] = [0.05, 0.25]
ranges4synth['kc_gt_T'] = [10, 60]

def synthesize_data(
    mt_sim_mode='isar2_c', seq_name='mt', 
    data_shape=(100, 10, 100), noise_level=NOISE_LEVEL, plot_noise_hist=False,
    add_bias=False, bias_location=10, bias_level=None
    ):
    # Sample parameter space:
    synthetic_datafeed = data.SlicesFeed.make_lh_sample( 
        seq_name=seq_name, 
        tissue_parameter_ranges_od=ranges4synth,
        shape=data_shape
    )   
    # Synthesize the signals:
    sim_mode = mt_sim_mode if seq_name=='mt' else 'expm_bmmat'
    synthetic_datafeed.slw = 10 if seq_name=='mt' else 1
    _, signal_3p = infer.infer(
        synthetic_datafeed, 
        simulation_mode=sim_mode
    )    
    #signal_3p *= (1.0 + noise_level * np.random.randn(*signal_3p.shape))  # adding noise to synthetic signals
    if type(noise_level) == list:
        noise_level = np.random.choice(NOISE_LEVEL, size=[1, *signal_3p.shape[1:]])        
        
    noise = noise_level * np.random.randn(*signal_3p.shape) / np.sqrt(signal_3p.shape[0])
    noise_norm = np.linalg.norm(noise, axis=0)
    if plot_noise_hist:
        plt.hist(noise_norm.flatten(), bins=50)
    signal_3p += noise
    if add_bias:
        bias_level = bias_level or np.random.choice(NOISE_LEVEL, size=[1, *signal_3p.shape[1:]])  # small bias of 1-3% of the signal amplitude
        signal_3p[bias_location:bias_location+10] += bias_level / np.sqrt(signal_3p.shape[0])
    # !! normalize the signals after adding noise and/or bias !!
    synthetic_datafeed.measured_normed_T = synthetic_datafeed.normalize(signal_3p)
    synthetic_datafeed.slw = 1
    return synthetic_datafeed, sim_mode, noise_norm

synthetic_datafeed_mt, mt_sim_mode, noise_norm = synthesize_data(mt_sim_mode=mt_sim_mode, plot_noise_hist=True) #, add_bias=False)
if do_amide:
    synthetic_datafeed_amide, _, _ = synthesize_data(seq_name='amide') 
else:
    synthetic_datafeed_amide = None

In [ ]:
synthetic_datafeed_mt_test, _, noise_norm = synthesize_data(mt_sim_mode=mt_sim_mode, add_bias=do_test_bias)
if do_amide:
    synthetic_datafeed_amide_test, _, _ = synthesize_data(seq_name='amide') 
else:
    synthetic_datafeed_amide_test = None

In [ ]:
ckpts = os.path.abspath(results_folder / 'ckpts') 

mt_train_folder = f'{ckpts}/mt'
cest_train_folder = f'{ckpts}/cest'

# !! retest only

print(f"mt_train_folder: {mt_train_folder}, cest_train_folder: {cest_train_folder}")

In [ ]:
import time 
print(time.ctime())

In [ ]:
from importlib import reload
from dataclasses import asdict

ensemble_size = 1  # MC-dropout
ensemble_size = 5  # Ensemble

ensemble_trainsteps_increment = 150

if do_train_supervised:
    shutil.rmtree(f'{ckpts}/supervised/mt_ens/', ignore_errors=True)    
    shutil.rmtree(f'{ckpts}/supervised/mt/', ignore_errors=True)
    
    # Disable the unsupervised-specific noise injection 
    train.train_config.tp_noise = False
    train.train_config.tpnoise_augmentation_burn_in = 0
    
    # Use dropout only for single model, not for ensemble (separate the two mechanisms for UQ via diversity of predictions):
    train.train_config.do_dropout = True if ensemble_size == 1 else False     
    
    train.train_config.use_shuffled_sampler = True    
    steps = 1000
    for jj in range(ensemble_size):
        if ensemble_size > 1:
            # Make sure initialization is random
            train.train_config.net_init_seed = int(time.time() * 1e9) % 2**32  # Random seed from time            
            if True:
                # Try to randomize all kinds of stuff to create diversity of ensemble - 
                # including the noise in the datafeed, to increase diversity of ensemble members.                                  
                synthetic_datafeed_mt, mt_sim_mode, noise_norm = \
                    synthesize_data(mt_sim_mode=mt_sim_mode, add_bias=True, bias_location=np.random.randint(0, 20), bias_level=0.1)
                synthetic_datafeed_mt.add_noise_to_signal = 0.02/np.sqrt(31)  # NOTE: a line above we just regenerate instead but let's push this as well
            # To make sure it's different, vary the early stopping 
            steps = ensemble_trainsteps_increment*(jj+1) # 200*(jj+1)  # np.random.choice([250, 500, 1000, 2000])      
        import logging
        logger = logging.getLogger(__name__)   
        synthetic_datafeed_mt.slw = 2
        model_state, loss_trend_mt, net_kwargs = \
                train.train(synthetic_datafeed_mt, model_state=None, 
                            pool2predict='c', logger=logger,
                            mode='reference_supervised', steps=steps, lr=3e-3)    
        tc_dict = asdict(train.train_config)
        tc_dict = {k: v for k, v in tc_dict.items() if type(v) != str}  # WA jax bug            
        ckpt_folder = f'{ckpts}/supervised/mt{"_ensemble/"+str(jj) if ensemble_size > 1 else "_single"}'
        net.save_ckpt(
            model_state,
            config={'train_cfg': tc_dict, 'net_cfg': net_kwargs},
            folder=ckpt_folder,
            step=666
        )
        print(f"Finished training ensemble member {jj+1}/{ensemble_size} with steps={steps}. Saved checkpoint to {ckpt_folder}.")
    predictor_mt = net.state2predictor(model_state)
elif do_train:
    t0 = time.time()
    shutil.rmtree(mt_train_folder, ignore_errors=True)
    if do_amide:
        shutil.rmtree(cest_train_folder, ignore_errors=True)
    synthetic_datafeed_mt.slw = 2
    predictor_mt, predictor_amide = pipelines.run_train(
        datafeed_mt=synthetic_datafeed_mt, 
        datafeed_cest=synthetic_datafeed_amide if do_amide else None, 
        ckpt_folder=ckpts,     
        do_cest=do_amide,
        mt_sim_mode = mt_sim_mode
    )
    train_time = time.time() - t0 
else:
    predictor_mt = net.load_ckpt(folder=mt_train_folder)[0]
    if do_amide:
        predictor_amide = net.load_ckpt(folder=cest_train_folder)[0]
    train_time = np.nan
    
# reconstruction error shown as 0 because not computed (to save time)

In [ ]:

#nnmodel
#net_kwargs

In [ ]:
import time 
print(time.ctime())

In [ ]:

def get_stochastic_predictor(model_state=None, ens_ckpts_folder=None):
    if ens_ckpts_folder is not None:
        model_states = [net.get_model_state(ens_ckpts_folder+f'{jj}/') for jj in range(ensemble_size)]
    else:
        assert model_state is not None, "Either model_state or ens_ckpts_folder must be provided"
        model_states = [model_state]    
    
    predictor_mt_l = [net.state2predictor(model_state, frozen_BN_infer=True) for model_state in model_states]
    pred_params = {'params': model_state.params, 'batch_stats': model_state.batch_stats}

    def stochastic_predictor(datafeed, key):
        _predictor = lambda batch: model_state.apply_fn(
            pred_params, batch, mutable=['batch_stats'],
            train=False, enable_dropout=True, rngs={'dropout': key} #, deterministic=False, # jax.random.PRNGKey(0)}
        )    
        mt_tissue_params_est, _mt_reconstructed_signal = infer.infer(
            datafeed, pool2predict='c',
            nn_predictor=_predictor, simulation_mode=mt_sim_mode, do_forward=False
        )
        f_est = 100 * mt_tissue_params_est["fc_T"]
        k_est = mt_tissue_params_est["kc_T"]
        return f_est, k_est
    
    return stochastic_predictor

if do_train_supervised:
    if ensemble_size == 1:
        stochastic_predictor = get_stochastic_predictor(model_state)
    else:
        # ensemble:
        ens_ckpts_folder = f'{ckpts}/supervised/mt_ensemble'        
        predictors_ensemble = [net.load_ckpt(ens_ckpts_folder+f'/{jj}', enable_dropout=False)[0] for jj in range(ensemble_size)]        


In [ ]:
# %history 
#f'{ckpts}/supervised/mt_ens'

In [ ]:
#!ls /home/alexf/phiVAE/figs/simulative/results_2026-05-14_20-15/ckpts/supervised/mt_ens

In [ ]:
def extract_mc_dropout_uncertainty(stochastic_predictor, datafeed, num_samples=10, use_vmap=True):
    """MC Dropout inference - run multiple stochastic forward passes to get uncertainty estimates
    """            
    keys = jax.random.split(jax.random.PRNGKey(0), num_samples)
    
    print('use_vmap', use_vmap)
    if use_vmap: # acceleration; had some past issues
        print('Using vmap for stochastic forward passes')
        single_pass = lambda key: stochastic_predictor(datafeed, key)
        f_est_samples, k_est_samples = jax.vmap(single_pass)(keys)            
    else:
        print('Using loop for stochastic forward passes')
        f_est_samples = []
        k_est_samples = []
        for jj in range(num_samples):
            f_est, k_est = stochastic_predictor(datafeed, keys[jj])        
            f_est_samples.append(f_est)
            k_est_samples.append(k_est)

    mean_f_est = np.mean(f_est_samples, axis=0)
    mean_k_est = np.mean(k_est_samples, axis=0)
    
    #fk_cov = np.cov(f_est_samples.reshape(num_samples, -1), k_est_samples.reshape(num_samples, -1), rowvar=False)  # Fails on memory
    fk_diff = np.concatenate([(f_est_samples-mean_f_est)[:, None, ...], 
                            (k_est_samples-mean_k_est)[:, None, ...]], axis=1)
    fk_cov = np.einsum('ij...,ik...->...jk', fk_diff, fk_diff) / (num_samples - 1)  # unbiased covariance estimate
    return (mean_f_est, mean_k_est, fk_cov, fk_diff)


def extract_ensemble_uncertainty(predictors_ensemble, datafeed):
    f_est_samples = []
    k_est_samples = []
    for _predictor in predictors_ensemble:
        mt_tissue_params_est, _mt_reconstructed_signal = infer.infer(
            datafeed, pool2predict='c',
            nn_predictor=_predictor, simulation_mode=mt_sim_mode, do_forward=False
        )
        f_est = 100 * mt_tissue_params_est["fc_T"]
        k_est = mt_tissue_params_est["kc_T"]
        f_est_samples.append(f_est)
        k_est_samples.append(k_est)

    mean_f_est = np.mean(f_est_samples, axis=0)
    mean_k_est = np.mean(k_est_samples, axis=0)
    
    fk_diff = np.concatenate([(f_est_samples-mean_f_est)[:, None, ...], 
                            (k_est_samples-mean_k_est)[:, None, ...]], axis=1)
    fk_cov = np.einsum('ij...,ik...->...jk', fk_diff, fk_diff) / (len(predictors_ensemble) - 1)  # unbiased covariance estimate
    return (mean_f_est, mean_k_est, fk_cov, fk_diff)


def cov2ellipse(cov_scaled, chisq=2.45):
    print('cov_scaled.shape', cov_scaled.shape)
    s2_scaled, u_scaled = np.linalg.eigh(cov_scaled)
    s_scaled = np.sqrt(s2_scaled)
    d0 = s_scaled[..., 0, None] * u_scaled[..., 0]
    d1 = s_scaled[..., 1, None] * u_scaled[..., 1]

    angle = np.degrees(np.arctan2(d1[..., 1], d1[..., 0]))
    chisq = 2.45  # Mahalanobis radius for 95% confidence in 2D
    height = np.linalg.norm(d1, axis=-1) * 2 * chisq
    width = np.linalg.norm(d0, axis=-1) * 2 * chisq
    return height, width, angle

if do_train_supervised:
    if ensemble_size == 1:
        f_est, k_est, cov_nnpred_scaled, fk_diff = extract_mc_dropout_uncertainty(stochastic_predictor, synthetic_datafeed_mt_test, num_samples=5, use_vmap=False) #0)
    else:
        f_est, k_est, cov_nnpred_scaled, fk_diff = extract_ensemble_uncertainty(predictors_ensemble, synthetic_datafeed_mt_test)

In [ ]:
synthetic_datafeed_mt.shape

In [ ]:
import importlib, inspect
import analyze_uncertainty as au
import analyze_uncertainty.core as aucore
import analyze_uncertainty.viz as auviz
import analyze_uncertainty.metrics as aumetrics

# Reload leaf modules first, then parent package re-exports.
for m in (aucore, auviz, aumetrics, au):
    importlib.reload(m)

print('viz_posteriors line:', inspect.getsourcelines(aumetrics.viz_posteriors)[1])

au.core.au_config.mt_sim_mode = mt_sim_mode
au.core.au_config.known_sigma = None # NOISE_LEVEL   # since we know the noise level in our synthetic data, let's use it for better likelihood estimation
au.core.au_config.known_gt = True             # ask for CDF@GT calculation as we have the ground truth in our synthetic data

num_samples = 30

datafeed2use = synthetic_datafeed_mt_test
if do_train_supervised:    
    if ensemble_size == 1:        
        f_est, k_est, cov_nnpred_scaled, fk_diff = extract_mc_dropout_uncertainty(stochastic_predictor, datafeed2use, num_samples=num_samples, use_vmap=False) #0)
    else:
        # datafeed2use = synthetic_datafeed_mt  # !! don't use test bias to make it easier for poor ensemble to get good coverage
        f_est, k_est, cov_nnpred_scaled, fk_diff = extract_ensemble_uncertainty(predictors_ensemble, datafeed2use) 
    f_gt = [100*datafeed2use.fc_gt_T[_x, 0, _y] for _x in range(100) for _y in range(100)]
    k_gt = [datafeed2use.kc_gt_T[_x, 0, _y] for _x in range(100) for _y in range(100)]
    reference = np.stack([f_gt, k_gt], axis=-1)  
    
    f_sigma = np.sqrt(cov_nnpred_scaled[..., 0, 0])
    k_sigma = np.sqrt(cov_nnpred_scaled[..., 1, 1])
    height, width, angle = cov2ellipse(cov_nnpred_scaled)   
else:
    transfer_res, _, __ = pipelines.transfer_and_plot(
        brain2test_mt=synthetic_datafeed_mt_test, brain2test_cest=None, 
        predictor_mt=predictor_mt, predictor_cest=predictor_amide if do_amide else None, 
        figsfolder=f'{results_folder}', figsfx='mt', do_slice_plots=None, do_boxplots=None,
        mt_simulation_mode=mt_sim_mode, do_forward=False
    )
    mt_tissue_params_est, amide_tissue_param_est = transfer_res[1], transfer_res[4]
    f_est, k_est, cov_nnpred_scaled, f_sigma, k_sigma, height, width, angle = \
        au.extract_nn_estimate_of_posterior(mt_tissue_params_est, synthetic_datafeed_mt_test.shape, is_cest=False)

In [ ]:
plt.hist(k_sigma.flatten(), bins=10)


In [ ]:
f_gt = [100*synthetic_datafeed_mt_test.fc_gt_T[_x, 0, _y] for _x in range(100) for _y in range(100)]
k_gt = [synthetic_datafeed_mt_test.kc_gt_T[_x, 0, _y] for _x in range(100) for _y in range(100)]
reference = np.stack([f_gt, k_gt], axis=-1)  

f_est_nn_l = [f_est[_x, 0, _y] for _x in range(100) for _y in range(100)]
k_est_nn_l = [k_est[_x, 0, _y] for _x in range(100) for _y in range(100)]
fk_est_nn = np.stack([f_est_nn_l, k_est_nn_l], axis=1)
cov_est_nn_arr = np.array([cov_nnpred_scaled[_x, 0, _y] for _x in range(100) for _y in range(100)])

In [ ]:
def plot_scatters(f_gt, k_gt, f_est_nn_l, k_est_nn_l, visualize=True):
    k_MAPE = np.mean(np.abs((k_gt - k_est_nn_l) / k_gt)) * 100
    f_MAPE = np.mean(np.abs((f_gt - f_est_nn_l) / f_gt)) * 100
    if visualize:
        plt.figure(figsize=(10, 2))
        plt.subplot(1, 2, 1)
        plt.scatter(f_gt, f_est_nn_l, marker='.', alpha=0.05)
        plt.plot([np.min(f_gt), np.max(f_gt)],[np.min(f_gt), np.max(f_gt)], 'k--')

        plt.title(f'MAPE: f={f_MAPE:.2f}%')
        plt.xlabel('Ground Truth f')
        plt.ylabel('NN estimate f')
        plt.subplot(1, 2, 2)
        plt.title(f'MAPE: k={k_MAPE:.2f}%')
        plt.scatter(k_gt, k_est_nn_l, marker='.', alpha=0.05)
        plt.plot([np.min(k_gt), np.max(k_gt)],[np.min(k_gt), np.max(k_gt)], 'k--')  
        plt.xlabel('Ground Truth k')
        plt.ylabel('NN estimate k')
    return f_MAPE, k_MAPE


plot_scatters(np.array(f_gt), np.array(k_gt), np.array(f_est_nn_l), np.array(k_est_nn_l))

In [ ]:
import os
os.makedirs(results_folder / 'intermediates', exist_ok=True)
npz_filename = str(results_folder / 'intermediates/data4calib_to_ref.npz')

np.savez_compressed(
    npz_filename,    
    cov_est_nn_arr=cov_est_nn_arr,
    reference=reference,
    fk_est_nn=fk_est_nn
)
print(npz_filename)

In [ ]:
import numpy as np
from scipy import stats

def get_mahalanobis_critical(coverage, dims):
    """
    Returns the Mahalanobis distance (M) for a specific coverage 
    level (e.g., 0.95) and dimension (k).
    """
    # M^2 follows a Chi-squared distribution with 'dims' degrees of freedom
    squared_m = stats.chi2.ppf(coverage, df=dims)
    return np.sqrt(squared_m)

coverage_levels = list(np.arange(0.2, 0.75, 0.1)) + list(np.arange(0.75, 1, 0.02)) # + [0.995, 0.999, 0.9999, 0.99999]
M_thresholds = [get_mahalanobis_critical(coverage, 2) for coverage in coverage_levels]
coverage_pct = 100.0 * np.array(coverage_levels, dtype=float)

# option for heuristic calibration (not used eventually, but could be useful for future work)
inflation = 1 # 

# Mahalanobises of the GROUND TRUTH under the NN-predicted posterior
gt_mahas = np.sqrt(((fk_est_nn - reference)[:, None, :] @ (np.linalg.inv(cov_est_nn_arr * inflation) @ (fk_est_nn - reference)[..., None])).squeeze())
empirical_coverage = np.array([np.sum(gt_mahas < mth) / len(gt_mahas) for mth in M_thresholds], dtype=float)
empirical_pct = 100.0 * empirical_coverage

list(zip(coverage_pct, empirical_pct))
plt.plot(coverage_pct, empirical_pct, marker='o')
plt.plot(coverage_pct, coverage_pct, 'k--')

In [ ]:
reload(au); reload(au.viz)
inflation = 1
nn_pred_dict = {'width': width*inflation, 'height': height*inflation, 'angle': angle, 'f': f_est, 'k': k_est, 'cov_nnpred_scaled': cov_nnpred_scaled}

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
        
for (_x, _y) in [(x, y) for x in range(2) for y in range(2)]:
    if do_train_supervised:
        num_samples = fk_diff.shape[0]
        ax = axes[_x, _y]
        gt = 100*datafeed2use.fc_gt_T[_x, 0, _y], datafeed2use.kc_gt_T[_x, 0, _y]
        (_1, _2, _3, nrmse, _df, _dk) = au.get_nrmse_grid(
            None, _x, _y, 0, datafeed2use, None, mt_sim_mode=mt_sim_mode,                
        )    
        au.viz.viz_posteriors_two_nn(
            ax, nrmse, nn_est1=None, nn_est2=None, gt=gt, # width1, height1, angle1, width2, height2, angle2, ax, 
            n_meas=31, _df=None, _dk=None, linewidth=1.5, markersize=5, is_cest=False, fontsize=14, do_heatmap=True, do_legend=False,
            #nn_colors = ("c",), nn_est_list=({k:v[_x, 0, _y] for k, v in nn_pred_dict.items()}, )
            nn_colors = ("c",)*(num_samples+1), 
            nn_est_list=[{k:v[_x, 0, _y] for k, v in nn_pred_dict.items()}] + \
                [{'f': f_est[_x, 0, _y]+fk_diff[jj, 0, _x, 0, _y], 'k': k_est[_x, 0, _y]+fk_diff[jj, 1, _x, 0, _y], 'width': 0, 'height': 0, 'angle': 0} for jj in range(num_samples)]
        )

In [ ]:
# The comparison to likelihood mapping below evaluates the self-supervised PS-VAE posteriors.
# For the supervised baselines (MC-dropout / ensemble, do_train_supervised=True) the run ends here;
# the exception halts both 'Run All' and papermill.
if do_train_supervised:
    print("Supervised-baseline run complete; skipping the comparison to likelihood mapping.")
    raise SystemExit

### Comparison to likelihood mapping on a subset of voxels

In [ ]:
au.au_config.loc_dict_res = 200
xy_points = [(x,x) for x in range(100)]
_cp_res = au.compare_posteriors(  
    xy_points, synthetic_datafeed_mt_test, mt_tissue_params_est,
    mt_sim_mode=mt_sim_mode, sli=0, voxels2sample=10, folder_name=str(results_folder / 'MT_voxels'), visualize=True,
    #known_sigmas=noise_norm, #[noise_norm[x,y] for (x,y) in xy_points]
)

In [ ]:
xy_points = [(x,y) for x in range(100) for y in range(num_test_voxels_x100)]
compare_posteriors_result = au.compare_posteriors(
    xy_points, synthetic_datafeed_mt_test, mt_tissue_params_est,
    mt_sim_mode=mt_sim_mode, sli=0, voxels2sample=1000, folder_name=str(results_folder / 'MT_voxels'), visualize=False
)

In [ ]:
import time 
print(time.ctime())

In [ ]:
k_est.shape

In [ ]:
(
    mahas1,
    mahas2,
    CR_areas,
    nnCR_areas,
    good_xy_points,
    modes_exact,
    means_exact,
    covs_exact,
    timings,
    exact_min_percs,
    nn_min_percs,
    extra_metrics_dict
    ) = compare_posteriors_result

f_best_dotprod_arr, k_best_dotprod_arr = zip(*means_exact)

In [ ]:
len(means_exact), len(k_est)

In [ ]:
results_folder

In [ ]:
f_gt = [100*synthetic_datafeed_mt_test.fc_gt_T[_x, 0, _y] for (_x, _y) in good_xy_points]
k_gt = [synthetic_datafeed_mt_test.kc_gt_T[_x, 0, _y] for (_x, _y) in good_xy_points]

In [ ]:
import os
os.makedirs(results_folder / 'intermediates', exist_ok=True)
npz_filename = str(results_folder / 'intermediates/MT_UQ_stats_allpoints.npz')

np.savez_compressed(
    npz_filename,
    mahas1=mahas1, mahas2=mahas2,
    CR_areas=CR_areas,  nnCR_areas=nnCR_areas,
    good_xy_points=good_xy_points, 
    f_est=f_est, k_est=k_est, cov_nnpred_scaled=cov_nnpred_scaled,
    covs_of_exact_posterior=covs_exact, modes_of_exact_posterior=modes_exact, means_of_exact_posterior=means_exact,
    timings=timings, train_time=0,
    exact_posterior_min_nrmse_percs=exact_min_percs, nn_posterior_min_nrmse_percs=nn_min_percs,
    f_gt=f_gt, k_gt=k_gt,
    **extra_metrics_dict
)

In [ ]:
plt.hist(mahas1, bins=np.arange(0, 5, 0.15), density=True)

In [ ]:
results_folder

### OLD analyses been here, see pre- 2026-Jun-01 